# Xpect AI — RAG Pipeline

This notebook builds the Retrieval-Augmented Generation (RAG) pipeline
for Xpect AI using the cleaned Netflix dataset.

## RAG Pipeline

Netflix Dataset
→ Data Preparation
→ Document Creation
→ Embeddings
→ Vector Store
→ Semantic Retrieval
→ LLM Generation

The cleaned Netflix dataset will be transformed into searchable
knowledge that the AI assistant can retrieve when answering user questions.

# Step by Step Impletation.

##### Imports

In [ ]:
import pandas as pd


#### Dataset and Basic info

In [2]:
input_path=("../Data/netflix_cleaned.csv")
df=pd.read_csv(input_path)
df.head()

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,Unknown,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,Unknown,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,Unknown,Unknown,Unknown,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,Unknown,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


In [3]:
df.isnull().sum()

show_id         0
type            0
title           0
director        0
cast            0
country         0
date_added      0
release_year    0
rating          0
duration        0
listed_in       0
description     0
dtype: int64

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype
---  ------        --------------  -----
 0   show_id       8807 non-null   str  
 1   type          8807 non-null   str  
 2   title         8807 non-null   str  
 3   director      8807 non-null   str  
 4   cast          8807 non-null   str  
 5   country       8807 non-null   str  
 6   date_added    8807 non-null   str  
 7   release_year  8807 non-null   int64
 8   rating        8807 non-null   str  
 9   duration      8807 non-null   str  
 10  listed_in     8807 non-null   str  
 11  description   8807 non-null   str  
dtypes: int64(1), str(11)
memory usage: 825.8 KB


In [5]:
df.columns.tolist()

['show_id',
 'type',
 'title',
 'director',
 'cast',
 'country',
 'date_added',
 'release_year',
 'rating',
 'duration',
 'listed_in',
 'description']

In [6]:
df.iloc[0]

show_id                                                        s1
type                                                        Movie
title                                        Dick Johnson Is Dead
director                                          Kirsten Johnson
cast                                                      Unknown
country                                             United States
date_added                                     September 25, 2021
release_year                                                 2020
rating                                                      PG-13
duration                                                   90 min
listed_in                                           Documentaries
description     As her father nears the end of his life, filmm...
Name: 0, dtype: object

In [7]:
df.iloc[0].to_dict()


{'show_id': 's1',
 'type': 'Movie',
 'title': 'Dick Johnson Is Dead',
 'director': 'Kirsten Johnson',
 'cast': 'Unknown',
 'country': 'United States',
 'date_added': 'September 25, 2021',
 'release_year': 2020,
 'rating': 'PG-13',
 'duration': '90 min',
 'listed_in': 'Documentaries',
 'description': 'As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.'}

##  Creating Movie Documents

Each Netflix title will be converted from a DataFrame row into a structured text document.

The document will contain the metadata and description needed for semantic retrieval.

Example:

Title: The Dark Knight
Type: Movie
Director: Christopher Nolan
Cast: Christian Bale, Heath Ledger
Country: United States
Release Year: 2008
Rating: PG-13
Duration: 152 min
Genres: Action, Crime, Drama
Description: ...

These documents will later be converted into embeddings and stored in a vector database.

In [8]:
def create_movie_document(row):
    document=f"""
Title : {row["title"]}
Type : {row["type"]}
Director : {row["director"]}
Cast : {row["cast"]}
Country : {row["country"]}
Release Year : {row["release_year"]}
Rating : {row["rating"]}
Duration : {row["duration"]}
Genres : {row["listed_in"]}
Description : {row["description"]}
"""
    return document
test_doc=create_movie_document(df.iloc[0])
print(test_doc)


Title : Dick Johnson Is Dead
Type : Movie
Director : Kirsten Johnson
Cast : Unknown
Country : United States
Release Year : 2020
Rating : PG-13
Duration : 90 min
Genres : Documentaries
Description : As her father nears the end of his life, filmmaker Kirsten Johnson stages his death in inventive and comical ways to help them both face the inevitable.



#### Creating Documents for entire Dataset

In [9]:
documents=df.apply(create_movie_document,axis=1).tolist()
print(f"Created {len(documents)} documents Sucessfully.")
documents[1]

Created 8807 documents Sucessfully.


'\nTitle : Blood & Water\nType : TV Show\nDirector : Unknown\nCast : Ama Qamata, Khosi Ngema, Gail Mabalane, Thabang Molaba, Dillon Windvogel, Natasha Thahane, Arno Greeff, Xolile Tshabalala, Getmore Sithole, Cindy Mahlangu, Ryle De Morny, Greteli Fincham, Sello Maake Ka-Ncube, Odwa Gwanya, Mekaila Mathys, Sandi Schultz, Duane Williams, Shamilla Miller, Patrick Mofokeng\nCountry : South Africa\nRelease Year : 2021\nRating : TV-MA\nDuration : 2 Seasons\nGenres : International TV Shows, TV Dramas, TV Mysteries\nDescription : After crossing paths at a party, a Cape Town teen sets out to prove whether a private-school swimming star is her sister who was abducted at birth.\n'

## Generating Text Embeddings

Embeddings convert each Netflix document into a numerical vector
that captures its semantic meaning.

We will use a pretrained SentenceTransformer model rather than
training an embedding model from scratch.

In [10]:
from sentence_transformers import SentenceTransformer
embedding_model = SentenceTransformer("all-MiniLM-L6-v2")

c:\Users\DELL\.venvs\xpect-ai\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 857.61it/s]


In [13]:
test_emd=embedding_model.encode(documents[0])
test_emd.shape

(384,)

#### EMBED ALL 8,807 MOVIES

In [14]:
embeddings=embedding_model.encode(
    documents,
    show_progress_bar=True,
)
print(embeddings.shape)

Batches: 100%|██████████| 276/276 [19:32<00:00,  4.25s/it]


(8807, 384)


#### Verify FAISS.

In [15]:
import faiss
print("FAISS imported Sucessfully")

FAISS imported Sucessfully


### Converting Embeddings to vector Matrix

In [16]:
import numpy as np
embedding_matrix=np.array(embeddings).astype("float32")
print("Embedding Matrix shape :",embedding_matrix.shape)

Embedding Matrix shape : (8807, 384)


#### Creating Faiss Index

In [17]:
dimensions=embedding_matrix.shape[1]
index=faiss.IndexFlatL2(dimensions)
print("Faiss Index Created sucessfully!")


Faiss Index Created sucessfully!


### Adding Embeddings matrix to index

In [18]:
index.add(embedding_matrix)
print("Vector Stored :",index.ntotal)

Vector Stored : 8807


##### Check Vector store exists

In [20]:
import os
os.makedirs("../vector_store",exist_ok=True)
print("Vector Store folder ready")

Vector Store folder ready


#### Save Faiss to index

In [21]:
faiss.write_index(
    index,
    "../vector_store/netflix_index"
)
print("Faiss Index saved sucessfully")

Faiss Index saved sucessfully


#### Save movie documents

In [22]:
import pickle
with open("../vector_store/documents.pkl","wb") as f:
    pickle.dump(documents,f)
print("Documents Saved sucessfully")


Documents Saved sucessfully


#### Save Movie Metadata

In [24]:
df.to_pickle("../vector_store/movies.pkl")
print("Movies Metadata  Sucessfully!")

Movies Metadata  Sucessfully!


#### Test Persistance actullay works or not

In [26]:
del index
loaded_index=faiss.read_index(
    "../vector_store/netflix_index"
)
print("Loaded Vectors :",loaded_index.ntotal)


Loaded Vectors : 8807


#### Test semantic Retrival

In [28]:
query="Old and Scary Movies"
query_embedding=embedding_model.encode(
    [query]).astype("float32")
print(query_embedding.shape)

(1, 384)


#### Search Faiss

In [29]:
distances,indices=loaded_index.search(
    query_embedding,
    5
)
print("Indices:",indices)
print("Distances:",distances)


Indices: [[7955 6938 7814 8290 6039]]
Distances: [[0.8904631  0.9082347  0.9822067  0.99828446 1.0002923 ]]


#### Retrival the Actual Movie

In [31]:
for idx in indices[0]:
    print(documents[idx])
    print("--"*80)



Title : Scary Movie
Type : Movie
Director : Keenen Ivory Wayans
Cast : Anna Faris, Jon Abrahams, Shannon Elizabeth, Shawn Wayans, Regina Hall, Marlon Wayans, Lochlyn Munro, Cheri Oteri, Carmen Electra, Dave Sheridan, Kurt Fuller, Rick Ducommun, James Van Der Beek, Keenen Ivory Wayans, Marissa Jaret Winokur, Dan Joffre
Country : United States
Release Year : 2000
Rating : R
Duration : 88 min
Genres : Comedies, Horror Movies
Description : The Wayans brothers spoof some of Hollywood's biggest blockbusters, including Scream, I Know What You Did Last Summer, The Matrix and American Pie.

----------------------------------------------------------------------------------------------------------------------------------------------------------------

Title : Haunters: The Art of the Scare
Type : Movie
Director : Jon Schnitzer
Cast : Unknown
Country : United States
Release Year : 2017
Rating : TV-MA
Duration : 89 min
Genres : Documentaries, Horror Movies
Description : This documentary takes us i

# RAG Retrieval Foundation Completed

The first phase 1 of the Xpect AI RAG pipeline has been successfully completed.

## What We Built

- Loaded the cleaned Netflix dataset
- Created structured movie documents
- Generated semantic embeddings using SentenceTransformers
- Created a FAISS vector index
- Stored all 8,807 movie embeddings
- Persisted the FAISS index to disk
- Saved movie documents and metadata
- Reloaded the vector store successfully
- Tested semantic similarity search
- Retrieved relevant movies using natural-language queries

## Vector Store

The persistent vector store contains:

```text
vector_store/
├── netflix.index
├── documents.pkl
└── movies.pkl